# N170 · 正确性证明与反例工作坊

**目标：** 能把直觉改写为完整论证并指出适用边界。

**先修：** N013, N043, N084, N093。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 循环不变量；交换论证；归纳；终止性；负例；证明与测试分工。

## 从问题到算法

正确性主张应该配一个可检查的证据。判断能否分成至多 groups 段时，成功不只返回 True，还给出每段的右开端点 cuts。验证器独立检查覆盖、非空、顺序和每段阈值。

非负数组的贪心让当前段尽量长。若另一个合法分段第一刀更早，可以把该刀右移到贪心位置：前段仍合法，被移走元素非负，所以后段负担不会增大。重复交换得到贪心所用段数最少。

加入负数后，“单个元素大于阈值必定无解”已经不成立。反例搜索只用来反驳缺少前提的推理，不把含负数数组传给不适用的生产函数再宣称算法正确。

## 最小实现

**本章接口：** `feasible_partition(nums, limit, groups)`、`verify_witness_partition(nums, cuts, limit)`、`find_small_greedy_counterexample()`

In [1]:
def feasible_partition(nums, limit, groups):
    """Nonnegative values; return (feasible, exclusive_end_cuts)."""
    if limit<0 or groups<1 or any(x<0 for x in nums):
        raise ValueError('Nonnegative values/limit and positive groups required')
    if not nums:return True,[]
    if max(nums)>limit:return False,[]
    cuts=[];total=0
    for i,value in enumerate(nums):
        if total+value>limit:
            cuts.append(i)
            total=0
        total+=value
    cuts.append(len(nums))
    return (True,cuts) if len(cuts)<=groups else (False,[])

def verify_witness_partition(nums, cuts, limit):
    """Witness validator also accepts signed values; no greedy inference is made."""
    if not nums:return cuts==[]
    if not cuts or cuts[-1]!=len(nums):return False
    previous=0
    for end in cuts:
        if not isinstance(end,int) or not previous<end<=len(nums):return False
        if sum(nums[previous:end])>limit:return False
        previous=end
    return previous==len(nums)

def find_small_greedy_counterexample():
    from itertools import product
    # The nonnegative proof uses 'a too-large singleton makes the task impossible'.
    # Negative neighbors can invalidate that inference, even with just one group.
    for n in range(1,4):
        for nums in product(range(-2,4),repeat=n):
            for limit in range(4):
                if any(x<0 for x in nums) and max(nums)>limit and sum(nums)<=limit:
                    return {'nums':list(nums),'limit':limit,'groups':1,
                            'invalid_greedy_claim':False,'valid_cuts':[n]}
    raise AssertionError('The enumerated domain should contain a counterexample')

## 正确性、前提与复杂度

交换论证的关键不是“选最长看起来好”，而是右移分割不会增加后续段的和，恰好用到了非负性。证据检查器只检查给定分段是否成立，不推断最少段数；因此可以用于独立核验，包括含负数的反例。

**代价：** 贪心与证据验证时间 O(n)，cuts 空间 O(groups)（构造阶段最坏 O(n)）。穷举检查只运行小数组，不放进正式求解路径。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

nums=[7,2,5,10,8]
ok,cuts=feasible_partition(nums,18,2)
start=0;rows=[]
for end in cuts:
    rows.append((start,end,nums[start:end],sum(nums[start:end])))
    start=end
show_table(['start','exclusive end','segment','sum'],rows)
show_table(['counterexample field','value'],list(find_small_greedy_counterexample().items()))

start,exclusive end,segment,sum
0,3,"[7, 2, 5]",14
3,5,"[10, 8]",18


counterexample field,value
nums,"[-2, 1]"
limit,0
groups,1
invalid_greedy_claim,False
valid_cuts,[2]


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
import itertools
ok,cuts=feasible_partition([7,2,5,10,8],18,2)
assert ok and len(cuts)<=2 and verify_witness_partition([7,2,5,10,8],cuts,18)
def brute_partition(nums,limit,groups):
    if not nums:return True
    for count in range(1,min(groups,len(nums))+1):
        for middle in itertools.combinations(range(1,len(nums)),count-1):
            cuts=list(middle)+[len(nums)]
            if verify_witness_partition(nums,cuts,limit):return True
    return False
for n in range(5):
    for nums in itertools.product(range(3),repeat=n):
        for limit in range(5):
            for groups in range(1,4):
                actual,cuts=feasible_partition(nums,limit,groups)
                assert actual==brute_partition(nums,limit,groups)
                if actual:assert len(cuts)<=groups and verify_witness_partition(nums,cuts,limit)
assert not verify_witness_partition([1,2],[0,2],3)
assert not verify_witness_partition([1,2],[1],3)
assert not verify_witness_partition([1,2],[2,2],3)
case=find_small_greedy_counterexample()
assert verify_witness_partition(case['nums'],case['valid_cuts'],case['limit'])
assert max(case['nums'])>case['limit']
try:feasible_partition(case['nums'],case['limit'],1)
except ValueError:pass
else:raise AssertionError('Signed array should be explicitly rejected')
print('N170: 所有本章断言通过')

N170: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 分别证明二分单调、区间贪心和Dijkstra。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 给出撤掉前提后失败的输入。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 410. Split Array Largest Sum（comparison）
- 435. Non-overlapping Intervals（comparison）：[官方入口](https://leetcode.com/problems/non-overlapping-intervals/)
- 743. Network Delay Time（comparison）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。